# 01 Crazyflie Simulation

This notebook can be used to simulate data generated by the crazyflie quadcopter using RotorPy.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from pipeline.noise import NoiseInjector
from pipeline.noise.core.noise_profile import NoiseProfile
from pipeline.synthetic import SyntheticSensorGenerator, SyntheticTimeOfFlightDistance, SyntheticUWBTransceiver
from rotorpy_simulation.crazyflie import Crazyflie, CrazyflieIMU
from rotorpy_simulation.crazyflie_simulation import CrazyflieSimulation

In [ ]:
SIMULATION_NAME: str = "01_crazyflie_circular"
DATA_PATH: str = "../../simulated_data/rotorpy/" + SIMULATION_NAME
plt.style.use('default')

## 1. Object Generation & Run
Instantiate the crazyflie and IMU builders, then run the full RotorPy flight simulation.

In [ ]:
from rotorpy_simulation.crazyflie.trajectories import get_circular_trajectory

crazyflie = Crazyflie()
imu_sensor = CrazyflieIMU()
trajectory = get_circular_trajectory()

simulation = CrazyflieSimulation(
    crazyflie_builder=crazyflie,
    imu_sensor_builder=imu_sensor,
    trajectory=trajectory)
simulation.simulation_duration = 20.0

result = simulation.run("01_crazyflie_circular")

## 2. Validation
Sanity-check the simulation result with a 3D trajectory plot and a ground-truth dataframe preview.

In [ ]:
gt = result.ground_truth
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot(gt.position[:, 0], gt.position[:, 1], gt.position[:, 2], color='royalblue', linewidth=1)
ax.scatter(*gt.position[0], color='green', s=80, marker='o', label='Start')
ax.scatter(*gt.position[-1], color='red', s=80, marker='x', label='End')
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_zlabel('Z (m)')
ax.set_title('Crazyflie Trajectory')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
ground_truth = result.ground_truth.to_dataframe()
ground_truth.head(n=10)

## 3. Synthetic Sensor Addition
Add sensors not modelled by RotorPy (ToF distance, UWB transceivers) by computing their readings from the ground-truth trajectory and attaching them to the result.

Verify the generated sensor data with the trajectory plot and a distance-over-time plot.

In [ ]:
tof_distance_noise_profile = NoiseProfile.from_yaml("../../pipeline/noise/profiles/VL53L1x_ToF-Distance.yaml")
uwb_transceiver_noise_profile = NoiseProfile.from_yaml("../../pipeline/noise/profiles/DWM1000_UWB_Transceiver.yaml")
tof_distance_noise = NoiseInjector(tof_distance_noise_profile)
uwb_transceiver_noise_0 = NoiseInjector(uwb_transceiver_noise_profile, seed=42, channels=[0])
uwb_transceiver_noise_1 = NoiseInjector(uwb_transceiver_noise_profile, seed=43, channels=[0])
uwb_transceiver_noise_2 = NoiseInjector(uwb_transceiver_noise_profile, seed=44, channels=[0])
uwb_transceiver_noise_3 = NoiseInjector(uwb_transceiver_noise_profile, seed=45, channels=[0])
uwb_transceiver_noise_4 = NoiseInjector(uwb_transceiver_noise_profile, seed=46, channels=[0])
uwb_transceiver_noise_5 = NoiseInjector(uwb_transceiver_noise_profile, seed=47, channels=[0])
uwb_transceiver_noise_6 = NoiseInjector(uwb_transceiver_noise_profile, seed=48, channels=[0])
uwb_transceiver_noise_7 = NoiseInjector(uwb_transceiver_noise_profile, seed=49, channels=[0])

# synthetic sensors, list of tuples (Sensor, NoiseInjector, DropoutInjector)
synthetic_sensors = [
    (SyntheticTimeOfFlightDistance(name="time_of_flight_distance"), tof_distance_noise, None),
    (SyntheticUWBTransceiver(name="uwb_transceiver_0", anchor_position=np.array([2.32, 1.39, 2.08]), anchor_id=0),
     uwb_transceiver_noise_1, None),
    (SyntheticUWBTransceiver(name="uwb_transceiver_1", anchor_position=np.array([2.32, -1.43, 2.08]), anchor_id=1),
     uwb_transceiver_noise_1, None),
    (SyntheticUWBTransceiver(name="uwb_transceiver_2", anchor_position=np.array([2.34, 1.39, 0.23]), anchor_id=2),
     uwb_transceiver_noise_2, None),
    (SyntheticUWBTransceiver(name="uwb_transceiver_3", anchor_position=np.array([2.32, -1.43, 0.2]), anchor_id=3),
     uwb_transceiver_noise_3, None),
    (SyntheticUWBTransceiver(name="uwb_transceiver_4", anchor_position=np.array([-2.32, 1.39, 2.25]), anchor_id=4),
     uwb_transceiver_noise_4, None),
    (SyntheticUWBTransceiver(name="uwb_transceiver_5", anchor_position=np.array([-2.32, -1.43, 1.87]), anchor_id=5),
     uwb_transceiver_noise_5, None),
    (SyntheticUWBTransceiver(name="uwb_transceiver_6", anchor_position=np.array([-2.32, 1.39, 0.23]), anchor_id=6),
     uwb_transceiver_noise_6, None),
    (SyntheticUWBTransceiver(name="uwb_transceiver_7", anchor_position=np.array([-2.32, -1.43, 0.24]), anchor_id=7),
     uwb_transceiver_noise_7, None),
]

generator: SyntheticSensorGenerator = SyntheticSensorGenerator(sensors=synthetic_sensors)
generator.apply(result)

print(f"all sensors: {result.sensors_clean.keys()} / {result.sensors_noisy.keys()}")

In [ ]:
from rotorpy_simulation.crazyflie_sim_validation_plots import plot_trajectory_3d, plot_uwb_distances

plot_trajectory_3d(result, synthetic_sensors)
plot_uwb_distances(result, synthetic_sensors)

## 4. Noise Injection
Apply a hardware-derived noise profile to a clean sensor stream to produce a realistic noisy version.

In [ ]:
# no noise to apply here -> all done within the simulation or when creating synthetic sensors.
# better to use simulation noise because it affects the controllers behavior.


## 5. Dropout Simulation
Simulate sensor outages by masking samples with NaN — either in fixed time windows or randomly across the stream.

In [ ]:
# no dropout applied during this simulation

## 6. Export
Write the simulation result to CSV files and a pickle for downstream use.

In [ ]:
result.export_csv_data(DATA_PATH)
result.save(DATA_PATH + "/" + SIMULATION_NAME + ".pkl")